<a href="https://colab.research.google.com/github/GODRATAN/FlyRank_ML-Intern/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GODRATAN/FlyRank_ML_Assignment/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

print(daily)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})


In [2]:
# Inspect the available March 2026 partition without filtering
files = daily["train"].cache_files
print("Number of dataset shards:", len(files))
print(files[:3])

Number of dataset shards: 39
[{'filename': '/root/.cache/huggingface/datasets/FlyRank___internship-warehouse/fact_content_daily_performance/0.0.0/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/internship-warehouse-train-00000-of-00039.arrow'}, {'filename': '/root/.cache/huggingface/datasets/FlyRank___internship-warehouse/fact_content_daily_performance/0.0.0/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/internship-warehouse-train-00001-of-00039.arrow'}, {'filename': '/root/.cache/huggingface/datasets/FlyRank___internship-warehouse/fact_content_daily_performance/0.0.0/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/internship-warehouse-train-00002-of-00039.arrow'}]


### Contract

- **Unit of analysis:** One row represents one content item for one client on one report date.
- **Time window:** March 2026 (`2026-03-01` through `2026-03-31`).
- **Lane:** Content Refresh Prioritization.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields

**Features**
- `gsc_impressions` — search visibility/exposure available at the decision moment.
- `gsc_clicks` — observed search clicks available at the decision moment.
- `gsc_avg_position` — observed average search position available at the decision moment.
- `ga4_pageviews` — observed page activity available at the decision moment.
- `ga4_engaged_sessions` — observed engagement available at the decision moment.

**Label / proxy**
- A future content-performance outcome for identifying pages that may need refresh prioritization.

**Context**
- `report_date` — identifies the observation date and supports the time window.
- `client_hash_id` — identifies the client for grouping and validation, not as a predictive feature.
- `content_hash_id` — identifies the content item and supports the grain, not as a predictive feature.
- `gsc_data_available` — indicates whether Search Console data is available.
- `ga4_data_available` — indicates whether Analytics data is available.

**Excluded**
- `client_hash_id` and `content_hash_id` are excluded from model features because they are identifiers, not meaningful predictive signals.
- Outcome-derived fields are excluded from features because they would leak information from the outcome into the prediction.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# Inspect the dataset metadata
print(daily["train"].info)

DatasetInfo(description='', citation='', homepage='', license='', features={'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessions': Value('int64'), 'ga4_users': Value('int64'), 'ga4_engaged_sessions': Value('int64'), 'ga4_total_engagement_sec': Value('int64'), 'sessions_organic': Value('int64'), 'sessions_direct': Value('int64'), 'sessions_referral': Value('int64'), 'sessions_social': Value('int64'), 'sessions_paid': Value('int64'), 'sessions_ai': Value('int64'), 'ai_chatgpt': Value('int64'), 'ai_perplexity': Value('int64'), 'ai_gemini': Value('int64'), 'ai_copilot': Value('int64'), 'ai_claude': Value('int64'), 'ai_me

In [4]:
# Loading only the March 2026 partition for ML-04 development
march = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    data_files={
        "train": "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
    },
    token=HF_TOKEN
)

print(march)

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 9841378
    })
})


In [5]:
# Query 1 — Grain verification
# One row represents one client × content item × report date.

df_march = march["train"].to_pandas()

grain_counts = (
    df_march
    .groupby(["report_date", "client_hash_id", "content_hash_id"])
    .size()
)

print("Number of duplicate grain combinations:", (grain_counts > 1).sum())
print("Maximum rows per grain:", grain_counts.max())

Number of duplicate grain combinations: 0
Maximum rows per grain: 1


In [6]:
# Query 2 — March 2026 row count and date span

print("March row count:", len(df_march))
print("Minimum report date:", df_march["report_date"].min())
print("Maximum report date:", df_march["report_date"].max())

March row count: 9841378
Minimum report date: 2026-03-01
Maximum report date: 2026-03-31


In [7]:
# Query 3 — Availability check
# Equivalent to filtering with: gsc_data_available IS TRUE

gsc_available = df_march[df_march["gsc_data_available"].eq(True)]

print("Rows with GSC data available:", len(gsc_available))
print("Rows surviving IS TRUE filter:", len(gsc_available))

Rows with GSC data available: 3611061
Rows surviving IS TRUE filter: 3611061


### Verification result

- **Grain:** No duplicate `report_date × client_hash_id × content_hash_id` combinations were observed; the maximum was 1 row per combination.
- **March slice:** 9,841,378 rows were observed from 2026-03-01 through 2026-03-31.
- **GSC availability:** 3,611,061 rows survived the `gsc_data_available IS TRUE` availability filter.

In [8]:
# ML-04 — Five-feature frame

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions",
]

features = df_march[
    ["report_date", "client_hash_id", "content_hash_id"] + feature_cols
].copy()

print("Feature frame shape:", features.shape)
print(features.head())

Feature frame shape: (9841378, 8)
  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   gsc_impressions  gsc_clicks  gsc_avg_position  ga4_pageviews  \
0               20           0          3.350000            NaN   
1                1           0          0.000000            NaN   
2              125           1          4.928000            NaN   
3                7           0          4.000000            NaN   
4               11           0          2.272727            NaN   

   ga4_engaged_sessions  
0                   NaN  
1                   NaN  
2                   NaN  
3                   NaN  
4                   NaN 

### Five features — available when?

- **`gsc_impressions`** — available at the decision moment because it is an observed Search Console performance measure.
- **`gsc_clicks`** — available at the decision moment because it records observed search clicks.
- **`gsc_avg_position`** — available at the decision moment because it is an observed Search Console ranking measure.
- **`ga4_pageviews`** — available at the decision moment because it records observed Analytics page activity.
- **`ga4_engaged_sessions`** — available at the decision moment because it records observed Analytics engagement.

In [9]:
# ML-04 — Deliberate leakage experiment
# Create a label-derived column on purpose.

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

leak_df = features.copy()

# Deliberately leaked "label-derived" signal:
# use whether the observed GSC click count is positive.
leak_df["leaked_label"] = (df_march["gsc_clicks"] > 0).astype(int)

X = leak_df[feature_cols]
y = leak_df["leaked_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Leaked experiment accuracy:", accuracy_score(y_test, pred))

# Removing the deliberately leaked column.
# The honest feature frame contains only the five planned features.

honest_features = features[feature_cols].copy()

print("Leaked column present:", "leaked_label" in honest_features.columns)
print("Honest feature count:", len(honest_features.columns))
print("Honest features:", list(honest_features.columns))

Leaked experiment accuracy: 1.0
Leaked column present: False
Honest feature count: 5
Honest features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


### Leakage conclusion

The deliberate leakage experiment produced an accuracy of 1.0 because the `leaked_label` was constructed directly from `gsc_clicks`, which was also supplied as a feature. This is an intentionally unrealistic result and demonstrates why label-derived information must not be used as a feature.

The leaked column was removed. The honest feature frame keeps only the five planned features.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

A key limitation of this data is **unbalanced client history**. Different clients have different amounts of usable Search Console and Analytics history, so comparisons across clients should be interpreted cautiously.

Some early rows can also be **GSC-only** because GA4 data was not yet available. Therefore, missing GA4 values should not automatically be interpreted as zero engagement.

The data also contains **overlapping time windows**, so features and outcomes must be aligned carefully to avoid using information from the outcome period.

This analysis provides measured, directional decision-support signals; it cannot establish that a content change caused a performance change.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.